# 11 — Receita acumulada das próximas 4 semanas (análise complementar)

**Pergunta:** quanto o vendedor deve receber nas próximas 4 semanas (o prazo de um boleto de fornecedor)?  
**Lê:** treino, série completa e teste pela trava (*secundaria*), `exogenas_semanais`, `selecao_modelo.json`.  
**Grava:** `acumulado_4_semanas` (artefato), `acumulado_4_semanas.csv` (usado pelo protótipo) e tabelas `acumulado_*`.

O resultado principal do TCC continua sendo **h = 1**. Aqui o modelo escolhido prevê as 4 semanas de forma **recursiva**: a previsão de cada semana entra no lugar do realizado para prever a seguinte. Variáveis macro ficam congeladas na origem; o calendário segue a semana-alvo. Não participa da seleção. Somar as previsões h = 1 do notebook 05 seria vazamento: cada uma usa o realizado da semana anterior.

In [ ]:
# === SETUP (idêntico em todos os notebooks; só muda NOTEBOOK) ===
NOTEBOOK = "11_acumulado_4_semanas"
RAIZ = "/content/drive/MyDrive/tcc_fluxo"      # Colab. Localmente: caminho da pasta do projeto (ou deixe None)

import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip install -q -r {RAIZ}/requirements.txt
except ImportError:                              # fora do Colab
    RAIZ = os.environ.get("TCC_RAIZ") or None
    if RAIZ is None:
        from pathlib import Path
        RAIZ = str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").is_file()))
sys.path.insert(0, f"{RAIZ}/src")
%load_ext autoreload
%autoreload 2

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tcc_fluxo import iniciar
C, A = iniciar(RAIZ, notebook=NOTEBOOK)

In [ ]:
from tcc_fluxo import acumulado as AC, Especificacao, graficos

H = 4
spec_json = A.carregar_json("painel_spec")
spec = Especificacao.de_json(spec_json, C)
MODELO = A.carregar_json("selecao_modelo")["modelo_escolhido"]
TEST_END = pd.Timestamp(A.carregar_json("corte")["TEST_END"])
train = A.carregar_treino()
serie = A.carregar_serie_completa(finalidade="secundaria")
exog = A.carregar_df("exogenas_semanais")
print(f"Modelo: {MODELO} | origens: {len(AC.origens(C.TRAIN_TARGET_END, TEST_END, H))}")

In [ ]:
longo = AC.rodar(train, serie, exog, MODELO, spec, C, TEST_END, H, macro_cols=spec_json["MACRO_FEATURES"])

# conferência: em h = 1 a recursão tem de reproduzir as previsões do notebook 05
prev05 = A.carregar_df("previsoes_teste")
h1 = longo[longo["h"] == 1].merge(prev05[["seller_id", "target_week", f"pred_{MODELO}"]],
                                  left_on=["seller_id", "semana_alvo"], right_on=["seller_id", "target_week"])
assert len(h1) == (longo["h"] == 1).sum() and np.allclose(h1["previsto"], h1[f"pred_{MODELO}"], rtol=0, atol=1e-9)
print("h = 1 idêntico ao notebook 05: OK")

larga = AC.tabela_larga(longo, serie, H)
A.salvar_df(larga, "acumulado_4_semanas")
A.tabela(larga, "acumulado_4_semanas.csv")

### Erro do acumulado de 4 semanas e erro por horizonte

In [ ]:
res, freq, por_h = AC.avaliar(larga, longo, MODELO)
A.tabela(res, "acumulado_metricas.csv"); A.tabela(por_h, "acumulado_erro_por_horizonte.csv")
A.salvar_json({"modelo": MODELO, "horizonte": H, **freq}, "acumulado_resumo")
display(res); display(por_h)
print(f"O acumulado do modelo ficou mais perto do realizado que o Naive × 4 em {freq['pct_mais_perto_que_naive4']:.0f}% "
      f"das {freq['n_comparacoes_sem_empate']:,} comparações vendedor-origem sem empate.")

In [ ]:
tot = larga.groupby("semana_fim")[["real_acum", "previsto_acum", "naive_acum"]].sum()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(tot.index, tot["real_acum"], color=graficos.INK, lw=2, label="Realizado")
axes[0].plot(tot.index, tot["previsto_acum"], color=graficos.C_CHOSEN, lw=2, label=f"{MODELO} (recursivo)")
axes[0].plot(tot.index, tot["naive_acum"], color=graficos.C_NAIVE, lw=2, ls="--", label="Naive × 4")
axes[0].set_title("Total da coorte nas 4 semanas seguintes (por semana final)", loc="left", fontsize=10)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v / 1000:,.0f} mil"))
axes[0].legend(frameon=False, fontsize=8)
axes[1].bar(por_h["h"].astype(str), por_h["WAPE"], color=graficos.C_CHOSEN)
for i, v in enumerate(por_h["WAPE"]):
    axes[1].text(i, v, f"{v:.0%}", ha="center", va="bottom", fontsize=9, color=graficos.INK2)
axes[1].set_title("WAPE por horizonte (vendedor-semana): a incerteza cresce", loc="left", fontsize=10)
axes[1].set_xlabel("semanas à frente")
for ax in axes:
    graficos.clean(ax)
fig.tight_layout(); A.figura(fig, "acumulado_4_semanas.png"); plt.show()

In [ ]:
A.finalizar()